# Lecture 3 companion — How a linear model learns

**Run it:** upload this file to Google Colab (nothing to install), or use the short [setup guide](../SETUP.md). This notebook imports only **NumPy** and **Matplotlib**; scikit-learn is not used.

## Goal

Trace one complete learning loop:

> data → prediction → residual → MSE → gradient → parameter update

We use **NumPy only** for the model. Matplotlib is used only to draw pictures.

## How to use this notebook

1. Read the question above a code cell.
2. Predict the answer with your partner.
3. Run the cell.
4. Explain what changed in plain language.

The plotting helper is collapsed because you do not need to understand it yet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.style.use("default")

## 1. A dataset is a matrix

Each **row** is one house. Each **column** is one feature.

- area is measured in units of 100 m²
- distance is measured in units of 10 km
- price is measured in units of $100,000

The fixed seed makes the same 16 houses every time.

In [ ]:
rng = np.random.default_rng(7)

area_m2 = rng.integers(60, 181, size=16).astype(float)
distance_km = rng.integers(1, 21, size=16).astype(float)

area = area_m2 / 100
distance = distance_km / 10
noise = rng.normal(0, 0.08, size=16)
price = 0.5 + 1.0 * area - 0.3 * distance + noise

X = np.column_stack([area, distance])
y = price

print("X shape:", X.shape)
print("y shape:", y.shape)
print("first 4 rows of X:\n", X[:4])
print("first 4 prices:", y[:4])

### Check before continuing

- What does `X[0, 0]` mean?
- What does `X[0, 1]` mean?
- Why is `X.shape` equal to `(16, 2)`?

In [ ]:
# Visualization helper: you do not need to understand this plotting code yet.
def plot_area_model(weight, bias, title):
    x_grid = np.linspace(area.min() - 0.05, area.max() + 0.05, 100)
    predictions = weight * x_grid + bias
    plt.figure(figsize=(7, 4))
    plt.scatter(area, price, s=55, color="#1D4ED8", label="houses")
    plt.plot(x_grid, predictions, color="#EA580C", linewidth=3, label="model")
    plt.xlabel("area (100 m² units)")
    plt.ylabel("price ($100k units)")
    plt.title(title)
    plt.legend()
    plt.show()


def plot_loss_history(loss_history):
    plt.figure(figsize=(7, 3.5))
    plt.plot(loss_history, color="#7C3AED", linewidth=2.5)
    plt.xlabel("training step")
    plt.ylabel("MSE")
    plt.title("The mistake score falls during training")
    plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
points = plt.scatter(area, price, c=distance, cmap="viridis", s=70)
plt.colorbar(points, label="distance (10 km units)")
plt.xlabel("area (100 m² units)")
plt.ylabel("price ($100k units)")
plt.title("Sixteen synthetic houses")
plt.show()

## 2. A model is a function with parameters

For one feature:

```text
prediction = weight × area + bias
```

- `weight` controls how much the prediction changes when area changes.
- `bias` is the prediction where the input is zero—the line's starting value.
- Together, they are the model's adjustable **parameters**.

In [ ]:
house_area = 1.2
weight = 1.0
bias = 0.5

prediction = weight * house_area + bias
print("predicted price ($100k units):", prediction)

### Predict before running

For the same house, which change moves the prediction more?

1. Add `0.1` to the weight.
2. Add `0.1` to the bias.

Explain your answer before checking it.

In [ ]:
original = weight * house_area + bias
change_weight = (weight + 0.1) * house_area + bias
change_bias = weight * house_area + (bias + 0.1)

print("original:          ", original)
print("weight + 0.1:      ", change_weight)
print("bias + 0.1:        ", change_bias)

In [ ]:
def predict_one_feature(inputs, weight, bias):
    return inputs * weight + bias


initial_weight = 0.4
initial_bias = 0.4
initial_predictions = predict_one_feature(area, initial_weight, initial_bias)

print("first 4 predictions:", initial_predictions[:4])
plot_area_model(initial_weight, initial_bias, "Our starting line")

### Interactive stop: control the line

Use the [Google Parameters Exercise](https://developers.google.com/machine-learning/crash-course/linear-regression/parameters-exercise).

Change one slider at a time. What does the **weight** do? What does the **bias** do?

## 3. Residuals and mean squared error

One prediction mistake is a **residual**:

```text
residual = prediction − actual value
```

Before learning the formula, try the [Google Loss Tutorial](https://developers.google.com/machine-learning/crash-course/linear-regression/loss). Which line seems better, and what should “better” mean?

### Hand calculation: three small houses

Use:

```text
x = [1, 2, 3]
y = [2, 3, 5]
weight = 1
bias = 0
```

On paper, calculate predictions, residuals, squared residuals, and their mean. Predict the MSE before running the next cell.

In [ ]:
x_small = np.array([1.0, 2.0, 3.0])
y_small = np.array([2.0, 3.0, 5.0])
w_small, b_small = 1.0, 0.0

pred_small = w_small * x_small + b_small
residual_small = pred_small - y_small
squared_residual_small = residual_small ** 2
loss_small = np.mean(squared_residual_small)

print("predictions:      ", pred_small)
print("residuals:        ", residual_small)
print("squared residuals:", squared_residual_small)
print("MSE:              ", loss_small)

In [ ]:
def mean_squared_error(actual, predicted):
    residuals = predicted - actual
    return np.mean(residuals ** 2)


initial_loss = mean_squared_error(y, initial_predictions)
print("MSE of the starting line:", initial_loss)

## 4. A gradient gives local update instructions

For one feature:

```text
weight gradient = average(2 × residual × input)
bias gradient   = average(2 × residual)
```

A positive gradient says increasing the parameter increases the loss nearby. A negative gradient says increasing it decreases the loss nearby.

We update in the opposite direction:

```text
new parameter = old parameter − learning rate × gradient
```

In [ ]:
gradient_w = np.mean(2 * residual_small * x_small)
gradient_b = np.mean(2 * residual_small)

print("weight gradient:", gradient_w)
print("bias gradient:  ", gradient_b)

### Direction check

Both gradients are negative. Before calculating anything, answer:

- Will the update increase or decrease the weight?
- Will it increase or decrease the bias?

In [ ]:
learning_rate = 0.1

new_w = w_small - learning_rate * gradient_w
new_b = b_small - learning_rate * gradient_b
new_predictions = new_w * x_small + new_b
new_loss = mean_squared_error(y_small, new_predictions)

print("new weight:", new_w)
print("new bias:  ", new_b)
print("old MSE:   ", loss_small)
print("new MSE:   ", new_loss)

### Interactive stop: walk downhill

Use the [UCLA Gradient Descent Visualizer](https://uclaacm.github.io/gradient-descent-visualiser/).

Before every move, point in the direction that should reduce the loss. The gradient is local advice, so it is recalculated after each step.

In [ ]:
candidate_weights = np.linspace(-0.5, 2.0, 150)
candidate_losses = [
    mean_squared_error(y_small, w * x_small + b_small)
    for w in candidate_weights
]

plt.figure(figsize=(7, 3.5))
plt.plot(candidate_weights, candidate_losses, color="#7C3AED", linewidth=3)
plt.scatter([w_small], [loss_small], color="#EA580C", s=90, label="starting weight")
plt.xlabel("weight")
plt.ylabel("MSE")
plt.title("A local loss landscape when bias is fixed")
plt.legend()
plt.show()

## 5. Training repeats the same learning step

Read each line and label it: **predict**, **loss**, **gradient**, or **update**.

In [ ]:
def gradients_one_feature(inputs, actual, weight, bias):
    predicted = weight * inputs + bias
    residuals = predicted - actual
    gradient_w = np.mean(2 * residuals * inputs)
    gradient_b = np.mean(2 * residuals)
    return gradient_w, gradient_b

In [ ]:
weight, bias = 0.4, 0.4
loss_history = []

for step in range(200):
    predicted = predict_one_feature(area, weight, bias)
    loss_history.append(mean_squared_error(y, predicted))
    gradient_w, gradient_b = gradients_one_feature(area, y, weight, bias)
    weight = weight - 0.05 * gradient_w
    bias = bias - 0.05 * gradient_b

print("learned weight:", weight)
print("learned bias:  ", bias)
print("final MSE:     ", loss_history[-1])

In [ ]:
plot_loss_history(loss_history)
plot_area_model(weight, bias, "Area-only model after training")

### Check

- Why does the curve fall quickly and then flatten?
- Why does the final line still miss some houses?
- What may happen if the learning rate is much too large?

One reason for the remaining pattern is that our synthetic prices also depend on distance.

## 6. Two features use a dot product

```text
prediction = area_weight × area + distance_weight × distance + bias
prediction = features @ weights + bias
```

The learning loop does not change. The model simply has one more adjustable weight.

In [ ]:
def predict(inputs, weights, bias):
    return inputs @ weights + bias


def gradients(inputs, actual, weights, bias):
    predicted = predict(inputs, weights, bias)
    residuals = predicted - actual
    gradient_weights = (2 / len(inputs)) * inputs.T @ residuals
    gradient_bias = 2 * np.mean(residuals)
    return gradient_weights, gradient_bias

In [ ]:
weights = np.zeros(2)
bias = 0.0
two_feature_losses = []

for step in range(500):
    predicted = predict(X, weights, bias)
    two_feature_losses.append(mean_squared_error(y, predicted))
    gradient_weights, gradient_bias = gradients(X, y, weights, bias)
    weights = weights - 0.05 * gradient_weights
    bias = bias - 0.05 * gradient_bias

print("learned [area, distance] weights:", weights)
print("learned bias:", bias)
print("final MSE:", two_feature_losses[-1])

In [ ]:
print("area-only final MSE:  ", loss_history[-1])
print("two-feature final MSE:", two_feature_losses[-1])

example_house = np.array([1.2, 0.5])
example_prediction = predict(example_house, weights, bias)
print("example prediction ($100k units):", example_prediction)

## 7. Essential vocabulary

- **fit:** use training examples to learn parameters.
- **predict:** use fixed learned parameters to produce outputs.
- **underfit:** the model misses an important pattern, even in training data.
- **overfit:** the model learns training details that do not help on new data.
- **validation data:** examples kept outside parameter updates to check choices on unseen cases.

We will study underfitting, overfitting, and validation more carefully when we train deeper networks.

## Final checks

Without looking back, answer:

1. What are the parameters of a one-feature linear model?
2. Why do we square residuals in MSE?
3. If a gradient is positive, which way does the update move the parameter?
4. Which five notebook lines form the learning loop?
5. Why might low training MSE fail to guarantee good predictions for new houses?

Return to the [Google Loss Tutorial](https://developers.google.com/machine-learning/crash-course/linear-regression/loss) and explain the better line using the words **prediction**, **residual**, and **loss**.

## Next step

In the next lecture, a linear prediction becomes a **neuron**. Several neurons plus nonlinear activation functions become a small MLP. The same learning loop remains.